# Gradients and Jacobians

**Goal:** Compute gradients for scalar fields and Jacobians for vector functions from scratch in PyTorch, validate against `torch.autograd` utilities, and show directional derivatives and the Hessian.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import torch


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure

device = configure()
print("running on:", device)


running on: mps


## Gradient of a Scalar Field: f: R^n -> R

The gradient `grad f(x) = [df/dx_1, ..., df/dx_n]^T` points in the direction of steepest increase.

We use `f(x) = 0.5 * ||x||^2 = 0.5 * sum(x_i^2)`, whose exact gradient is `grad f(x) = x`.

We build the gradient component-by-component via central finite differences, then compare with autograd.

In [2]:
def finite_diff_gradient(f, x: torch.Tensor, h: float = 1e-5) -> torch.Tensor:
    """Gradient of scalar f: R^n -> R via central finite differences.

    Uses float64 on cpu for accuracy.
    """
    x_cpu = x.detach().cpu().double()
    grad = torch.zeros_like(x_cpu)
    for j in range(x_cpu.numel()):
        xph = x_cpu.clone()
        xph[j] += h
        xmh = x_cpu.clone()
        xmh[j] -= h
        grad[j] = (f(xph) - f(xmh)) / (2.0 * h)
    return grad


def f_scalar_field(x: torch.Tensor) -> torch.Tensor:
    """f(x) = 0.5 * ||x||^2.  Exact gradient: x."""
    return 0.5 * (x ** 2).sum()


# Evaluate at x = [1, 2, 3]
x_test = torch.tensor([1.0, 2.0, 3.0], device=device)
fd_grad = finite_diff_gradient(f_scalar_field, x_test)

# Autograd gradient
x_ag = torch.tensor([1.0, 2.0, 3.0], device=device, requires_grad=True)
out = f_scalar_field(x_ag)
grad_ag, = torch.autograd.grad(out, x_ag)

print(f"Finite-diff gradient: {fd_grad.tolist()}")
print(f"Autograd gradient:    {grad_ag.cpu().tolist()}")
print(f"Exact (x itself):     {x_test.cpu().tolist()}")

assert torch.allclose(fd_grad.float(), grad_ag.cpu().float(), atol=1e-4),     "Finite-diff gradient does not match autograd!"
print("Assertion passed: finite-diff gradient matches autograd ✓")


Finite-diff gradient: [1.0000000000065512, 2.0000000000131024, 3.000000000019653]
Autograd gradient:    [1.0, 2.0, 3.0]
Exact (x itself):     [1.0, 2.0, 3.0]
Assertion passed: finite-diff gradient matches autograd ✓


## Jacobian of a Vector Function: F: R^n -> R^m

The Jacobian `J[i, j] = dF_i/dx_j` has shape `(m, n)`.

We use `F(x1, x2) = [x1^2 + x2,  x1*x2]`, so:
```
J_F = [[2*x1,  1 ],
        [ x2, x1]]
```

**Row-by-row construction:** each row `i` of the Jacobian is the gradient of output `F_i`.

In [3]:
def F_vector(x: torch.Tensor) -> torch.Tensor:
    """F(x1, x2) = [x1^2 + x2, x1*x2].  Jacobian shape (2, 2)."""
    return torch.stack([x[0] ** 2 + x[1], x[0] * x[1]])


def jacobian_row_by_row(F, x: torch.Tensor) -> torch.Tensor:
    """Jacobian of F: R^n -> R^m via per-output autograd gradients.

    Each row i is the gradient of F_i with respect to x.
    """
    # cpu float32 — mps handles this fine; float64 not needed here
    x_cpu = x.detach().cpu().float().requires_grad_(True)
    outputs = F(x_cpu)
    rows = []
    for i in range(outputs.numel()):
        grad_i, = torch.autograd.grad(outputs[i], x_cpu, retain_graph=True)
        rows.append(grad_i)
    return torch.stack(rows)  # shape (m, n)


x_jac = torch.tensor([1.0, 2.0], device=device)
J_manual = jacobian_row_by_row(F_vector, x_jac)

# Exact: J = [[2*1, 1], [2, 1]]
J_exact = torch.tensor([[2.0, 1.0], [2.0, 1.0]])

print("Row-by-row Jacobian at (1, 2):")
print(J_manual)
print("\nExact Jacobian at (1, 2):")
print(J_exact)

assert torch.allclose(J_manual, J_exact, atol=1e-5), "Row-by-row Jacobian != exact!"
print("\nAssertion passed: row-by-row Jacobian matches exact formula ✓")


Row-by-row Jacobian at (1, 2):
tensor([[2., 1.],
        [2., 1.]])

Exact Jacobian at (1, 2):
tensor([[2., 1.],
        [2., 1.]])

Assertion passed: row-by-row Jacobian matches exact formula ✓


## Idiomatic: `torch.autograd.functional.jacobian`

PyTorch provides `torch.autograd.functional.jacobian` for full Jacobian computation.
We validate that row-by-row matches it exactly.  
Note: runs on cpu — `jacobian` may have MPS limitations with certain ops.

In [4]:
# torch.autograd.functional.jacobian — cpu tensor for mps compatibility
x_jac_cpu = torch.tensor([1.0, 2.0])

J_autograd = torch.autograd.functional.jacobian(F_vector, x_jac_cpu)  # shape (2, 2)

print("torch.autograd.functional.jacobian:")
print(J_autograd)

assert torch.allclose(J_manual.cpu(), J_autograd, atol=1e-5),     "Row-by-row Jacobian does not match torch.autograd.functional.jacobian!"
print("\nAssertion passed: row-by-row Jacobian matches autograd.functional.jacobian ✓")

assert torch.allclose(J_autograd, J_exact, atol=1e-6),     "Autograd Jacobian does not match exact!"
print("Assertion passed: both Jacobians match exact formula ✓")


torch.autograd.functional.jacobian:
tensor([[2., 1.],
        [2., 1.]])

Assertion passed: row-by-row Jacobian matches autograd.functional.jacobian ✓
Assertion passed: both Jacobians match exact formula ✓


## Directional Derivative

The directional derivative of `f` in direction `v` (unit vector) is:
```
D_v f(x) = grad f(x)^T v
```
It measures how fast `f` increases in direction `v`.
The gradient direction gives the maximum rate of increase, equal to `||grad f||`.

In [5]:
# Use f(x) = 0.5 * ||x||^2 with gradient = x
x_dir = torch.tensor([3.0, 4.0], device=device, requires_grad=True)
out_dir = f_scalar_field(x_dir)
grad_dir, = torch.autograd.grad(out_dir, x_dir)

# Unit direction v = [1/sqrt(2), 1/sqrt(2)]
v = torch.tensor([1.0, 1.0], device=device) / (2.0 ** 0.5)

# Directional derivative = grad . v
dir_deriv = (grad_dir * v).sum()

# Along the gradient itself (maximum increase direction)
grad_hat = grad_dir / grad_dir.norm()
max_dir_deriv = (grad_dir * grad_hat).sum()

print(f"x = {x_dir.detach().cpu().tolist()}")
print(f"grad f(x) = {grad_dir.cpu().tolist()}")
print(f"\nDirection v = [1/sqrt(2), 1/sqrt(2)]:")
print(f"  D_v f(x) = grad f . v = {dir_deriv.item():.6f}")
print(f"\nAlong gradient direction (max increase):")
print(f"  D_ghat f(x) = ||grad f|| = {max_dir_deriv.item():.6f}")
print(f"  ||grad f||              = {grad_dir.norm().item():.6f}")

assert torch.allclose(max_dir_deriv, grad_dir.norm(), atol=1e-5),     "Max directional derivative != ||grad||!"
print("Assertion passed: max directional derivative equals ||grad f|| ✓")


x = [3.0, 4.0]
grad f(x) = [3.0, 4.0]

Direction v = [1/sqrt(2), 1/sqrt(2)]:
  D_v f(x) = grad f . v = 4.949747

Along gradient direction (max increase):
  D_ghat f(x) = ||grad f|| = 5.000000
  ||grad f||              = 5.000000
Assertion passed: max directional derivative equals ||grad f|| ✓


## Optional: Hessian via `torch.autograd.functional.hessian`

The Hessian `H[i, j] = d^2 f / (dx_i dx_j)` is the Jacobian of the gradient.  
For `f(x) = 0.5 * ||x||^2` the Hessian is the identity matrix `I`.  
Note: `hessian` requires cpu tensors — MPS does not support the nested grad needed.

In [6]:
# Hessian requires cpu; mps does not support create_graph=True for all ops
x_hess = torch.tensor([3.0, 4.0])  # cpu tensor

H = torch.autograd.functional.hessian(f_scalar_field, x_hess)  # shape (2, 2)

print("Hessian of f(x) = 0.5*||x||^2 at any x:")
print(H)

I = torch.eye(2)
assert torch.allclose(H, I, atol=1e-6), "Hessian != Identity for 0.5||x||^2!"
print("Assertion passed: Hessian of 0.5*||x||^2 is the identity matrix ✓")
print("(Constant Hessian means f is perfectly quadratic -- curvature is uniform.)")


Hessian of f(x) = 0.5*||x||^2 at any x:
tensor([[1., 0.],
        [0., 1.]])
Assertion passed: Hessian of 0.5*||x||^2 is the identity matrix ✓
(Constant Hessian means f is perfectly quadratic -- curvature is uniform.)


## Memory Implications of Full Jacobians

For a layer `F: R^n -> R^m`, the full Jacobian has `m x n` entries.

| n (inputs) | m (outputs) | Jacobian entries | float32 memory |
| --- | --- | --- | --- |
| 512 | 512 | 262 144 | 1 MB |
| 4 096 | 4 096 | 16 777 216 | 64 MB |
| 32 768 | 32 768 | 1 073 741 824 | 4 GB |

Backpropagation avoids materializing the Jacobian by using **vector-Jacobian products** (VJPs):  
`v^T J` costs `O(n + m)` memory instead of `O(m * n)`.

In [7]:
for n in [64, 512, 2048]:
    m = n
    params = m * n
    mb = params * 4 / 1e6
    print(f"  Layer ({n:5d},{m:5d}): {params:>12,} Jacobian entries  = {mb:>8.1f} MB")

print("\nVJP (v^T J) only needs to store the output vector, not the full Jacobian.")


  Layer (   64,   64):        4,096 Jacobian entries  =      0.0 MB
  Layer (  512,  512):      262,144 Jacobian entries  =      1.0 MB
  Layer ( 2048, 2048):    4,194,304 Jacobian entries  =     16.8 MB

VJP (v^T J) only needs to store the output vector, not the full Jacobian.


## Takeaways

- **Gradient** `grad f(x)` is an n-vector; the gradient of `0.5*||x||^2` is `x` (and its Hessian is `I`).
- **Jacobian** `J_F(x)` is an `(m, n)` matrix; build it row-by-row from per-output autograd gradients.
- `torch.autograd.functional.jacobian` gives the full Jacobian; row-by-row from `autograd.grad` is equivalent.
- **Directional derivative** `D_v f(x) = grad f . v`; maximum is achieved along the gradient direction, magnitude `||grad f||`.
- **Hessian** `H = J(grad f)` is `(n, n)`; for quadratic `f` it is constant.  
  Use `torch.autograd.functional.hessian` — requires CPU tensors.
- **Memory:** full Jacobians for large layers are prohibitive; backprop uses VJPs (`O(n+m)`) instead of `O(m*n)`.
